In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [13]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [14]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [15]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [16]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [17]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [18]:

initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': 'This outline aims to create a comprehensive, engaging, and informative blog post about the rise of AI in India, targeting a broad audience from tech enthusiasts to policymakers and investors.\n\n---\n\n## Blog Title Ideas:\n*   India\'s AI Revolution: Powering Progress and Shaping the Future\n*   From Silicon Valley to Silicon India: Charting the Rise of AI in the Subcontinent\n*   The AI Awakening: How India is Embracing and Innovating with Artificial Intelligence\n*   Beyond the Hype: Understanding the Real Impact of AI in India\n*   India\'s Digital Destiny: The Unstoppable Ascent of AI\n\n---\n\n## **Detailed Blog Outline: The Rise of AI in India**\n\n**Target Audience:** Tech enthusiasts, industry professionals, investors, policymakers, students, and general readers interested in technology and India\'s economic development.\n\n**Tone:** Informative, optimistic, balanced, and forward-looking.\n\n---\n\n### **I. Introduction (Approx. 150